In [ ]:
# === PACKAGE THE PREDICTED STRUCTURES FOR DOWNLOAD (CPU ONLY, NO GPU QUOTA) ===## Every structure this project has predicted lives only inside a Kaggle notebook's output# and has never been looked at. Roughly 250 complexes, and every selection decision so far# was made from numbers in a table. This pulls the 20 submitted TNF designs into one small# zip so they can be downloaded in a single click and actually examined.## It SEARCHES for the files by name across all attached inputs rather than assuming a# directory layout. A guess about where Kaggle mounts a notebook's output would fail# silently and produce an empty zip that looks like a successful run.import glob, json, os, shutil, zipfile# The 12 switches are named <parent>__<substitution> by the burial run; the 8 controls are# the bare parent sequences, whose complexes come from the generation rounds.SWITCHES = {    "tnf_sw_01": "tnf_w0_012__A28H",  "tnf_sw_02": "tnf2_w1_001__L13H",    "tnf_sw_03": "tnf3_w0_017__Y18H", "tnf_sw_04": "tnf_w0_011__L18H",    "tnf_sw_05": "tnf2_w0_001__Y37H", "tnf_sw_06": "tnf2_w0_007__L30H",    "tnf_sw_07": "tnf3_w1_009__I22H", "tnf_sw_08": "tnf3_w0_012__E54H",    "tnf_sw_09": "tnf3_w0_016__L9H",  "tnf_sw_10": "tnf3_w0_007__E56H",    "tnf_sw_11": "tnf2_w0_008__E2H",  "tnf_sw_12": "tnf_w0_007__Q46H",}CONTROLS = {    "tnf_ct_01": "tnf_w0_012",  "tnf_ct_02": "tnf2_w1_001",    "tnf_ct_03": "tnf3_w0_017", "tnf_ct_04": "tnf_w0_011",    "tnf_ct_05": "tnf2_w0_001", "tnf_ct_06": "tnf2_w0_007",    "tnf_ct_07": "tnf3_w1_009", "tnf_ct_08": "tnf3_w0_012",}TARGETS = ["TNF_site_crop.pdb", "TNF_mouse_site_crop.pdb"]found = {}for path in glob.glob("/kaggle/input/**/*.pdb", recursive=True):    base = os.path.basename(path)[:-4]    # Prefer the largest copy when a name appears in more than one attached output: a    # truncated or partial write would otherwise win on an arbitrary glob ordering.    if base not in found or os.path.getsize(path) > os.path.getsize(found[base]):        found[base] = pathprint("scanned attached inputs: %d distinct pdb basenames" % len(found), flush=True)out = "/kaggle/working/viewer"if os.path.exists(out):    shutil.rmtree(out)os.makedirs(out)manifest, missing = [], []for name, key in sorted(SWITCHES.items()) + sorted(CONTROLS.items()):    role = "switch" if name in SWITCHES else "control"    if key in found:        shutil.copy(found[key], "%s/%s.pdb" % (out, name))        manifest.append(dict(name=name, role=role, key=key,                             bytes=os.path.getsize(found[key])))    else:        missing.append((name, role, key))targets_got = []for t in TARGETS:    if t[:-4] in found:        shutil.copy(found[t[:-4]], "%s/%s" % (out, t))        targets_got.append(t)for name, role, key in missing:    print("MISSING %-10s %-8s %s" % (name, role, key), flush=True)print("\ncopied %d of 20 designs, %d of %d targets"      % (len(manifest), len(targets_got), len(TARGETS)), flush=True)json.dump(dict(designs=manifest, targets=targets_got,               missing=[dict(name=n, role=r, key=k) for n, r, k in missing]),          open("%s/manifest.json" % out, "w"), indent=1)zpath = "/kaggle/working/tnf_structures.zip"with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:    for f in sorted(os.listdir(out)):        z.write("%s/%s" % (out, f), f)print("wrote %s  (%.2f MB)" % (zpath, os.path.getsize(zpath) / 1e6), flush=True)if not manifest:    raise SystemExit("nothing was packaged - the inputs attached do not contain these "                     "structures. Check which notebook outputs are attached.")if missing:    print("\nNote: %d design(s) missing above. The zip still contains the rest; this is "          "not a failed run, but the viewer will have gaps." % len(missing), flush=True)